In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers datasets pandas torch sentencepiece rouge-score scikit-learn matplotlib seaborn -q

## 1. Load Dataset

In [ ]:
import pandas as pd
file_path = "/content/drive/MyDrive/cloth/amazon-fashion-800k+-user-reviews-dataset.csv"
df = pd.read_csv(file_path)
print(df.shape)
print(df["target"].value_counts())
print(df[["text", "target"]].head())


## 2. Slang & Abbreviation Normalisation

Reviews contain informal slang (`tbh`, `tts`, `fire`, `mid`, etc.) that standard tokenisers treat as noise. We map them to clean English **before** any other preprocessing so the model trains on consistent signal.

In [1]:
import re

SLANG_MAP = {
    # ── Sentiment slang ───────────────────────────────────────────
    r"\btrash\b":          "terrible",
    r"\bgarbage\b":        "terrible",
    r"\bjunk\b":           "poor quality",
    r"\bfire\b":           "excellent",
    r"\blit\b":            "excellent",
    r"\bgoat\b":           "greatest of all time",
    r"\bslay(ing)?\b":     "excellent",
    r"\bbanger\b":         "great product",
    r"\bmid\b":            "mediocre",
    r"\bdogwater\b":       "terrible",
    r"\bwack\b":           "bad",
    r"\bripoff\b":         "overpriced",
    r"\brip off\b":        "overpriced",
    r"\bscam\b":           "fraudulent",
    r"chef'?s kiss":       "perfect",
    r"\b10/10\b":          "perfect",
    r"\b5/5\b":            "perfect",
    r"\b1/5\b":            "terrible",
    # ── Internet abbreviations ────────────────────────────────────
    r"\btbh\b":            "to be honest",
    r"\bngl\b":            "not gonna lie",
    r"\bimo\b":            "in my opinion",
    r"\bimho\b":           "in my humble opinion",
    r"\bfwiw\b":           "for what it is worth",
    r"\bsmh\b":            "shaking my head",
    r"\bomg\b":            "oh my god",
    r"\blol\b":            "laughing",
    r"\bwtf\b":            "what the heck",
    r"\bbtw\b":            "by the way",
    r"\bfyi\b":            "for your information",
    r"\bafaik\b":          "as far as i know",
    # ── Fit / size slang ─────────────────────────────────────────
    r"\btts\b":            "true to size",
    r"\brun small\b":      "runs small",
    r"\brun big\b":        "runs large",
    r"\bsize up\b":        "order a size larger",
    r"\bsize down\b":      "order a size smaller",
    # ── Fabric / comfort ─────────────────────────────────────────
    r"\bsuper comfy\b":    "very comfortable",
    r"\bsuper soft\b":     "very soft",
    r"\bpaper thin\b":     "very thin fabric",
    r"\bsee[ -]?through\b":"transparent fabric",
    r"\bfell apart\b":     "broke apart",
    r"\bfeel apart\b":     "broke apart",
    # ── Delivery slang ───────────────────────────────────────────
    r"\bfast af\b":        "very fast",
    r"\bquick af\b":       "very quick",
    # ── Contractions ─────────────────────────────────────────────
    r"\bwon't\b":  "will not",  r"\bcan't\b":    "cannot",
    r"\bdon't\b":  "do not",    r"\bdoesn't\b":  "does not",
    r"\bdidn't\b": "did not",   r"\bisn't\b":    "is not",
    r"\bwasn't\b": "was not",   r"\baren't\b":   "are not",
    r"\bwouldn't\b":"would not", r"\bshouldn't\b":"should not",
    r"\bcouldn't\b":"could not", r"\bhaven't\b":  "have not",
    r"\bhasn't\b": "has not",
    r"\bI'm\b": "I am",   r"\bI've\b": "I have",
    r"\bI'll\b": "I will", r"\bI'd\b":  "I would",
    r"\bit's\b": "it is",  r"\bthat's\b":"that is",
    r"\bthey're\b":"they are", r"\bwe're\b": "we are",
    r"\byou're\b": "you are",  r"\bwhat's\b":"what is",
    # ── Elongated words:  soooo -> so ─────────────────────────────
    r"(\w)\1{2,}": r"\1\1",
}

def normalise_slang(text: str) -> str:
    text = text.lower()
    for pattern, replacement in SLANG_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

# Quick test
samples = [
    "tbh this is trash, fell apart after 1 wash smh",
    "omg soooo soft, tts, 10/10 would buy again",
    "mid quality, paper thin fabric, ripoff for the price",
    "slaying in this dress, fire fit and the color is lit",
    "ngl the stitching is weak af, returned it immediately",
    "goat level comfort, super comfy and design is chef's kiss",
]
for s in samples:
    print(f"IN : {s}\nOUT: {normalise_slang(s)}\n")


IN : tbh this is trash, fell apart after 1 wash smh
OUT: to be honest this is terrible, broke apart after 1 wash shaking my head

IN : omg soooo soft, tts, 10/10 would buy again
OUT: oh my god soo soft, true to size, perfect would buy again

IN : mid quality, paper thin fabric, ripoff for the price
OUT: mediocre quality, very thin fabric fabric, overpriced for the price

IN : slaying in this dress, fire fit and the color is lit
OUT: excellent in this dress, excellent fit and the color is excellent

IN : ngl the stitching is weak af, returned it immediately
OUT: not gonna lie the stitching is weak af, returned it immediately

IN : goat level comfort, super comfy and design is chef's kiss
OUT: greatest of all time level comfort, very comfortable and design is perfect



## 3. Text Cleaning

In [ ]:
def clean_text(text):
    text = str(text)
    text = normalise_slang(text)                            # slang first
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)             # remove non-ASCII
    text = re.sub(r'http\S+|www\.\S+', ' ', text)          # remove URLs
    text = re.sub(r"[^a-zA-Z0-9 .,!?'\-]", ' ', text)     # keep useful punct
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df[['text', 'target']].dropna()
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 20]

sentiment_map = {1: 'positive', 0: 'neutral', -1: 'negative'}
df['sentiment'] = df['target'].map(sentiment_map)

print(df['sentiment'].value_counts())
print(f"Total usable reviews: {len(df)}")


## 4. Aspect Extraction

In [ ]:
ASPECT_KEYWORDS = {
    'fabric':    ['fabric', 'material', 'cloth', 'cotton', 'silk', 'polyester',
                  'linen', 'texture', 'soft', 'rough', 'thin', 'thick', 'breathable',
                  'very thin fabric', 'transparent fabric'],
    'fit':       ['fit', 'fitted', 'fitting', 'loose', 'tight', 'snug', 'baggy', 'oversized'],
    'size':      ['size', 'sizing', 'small', 'large', 'medium', 'runs small',
                  'runs large', 'true to size', 'order a size'],
    'stitching': ['stitch', 'stitching', 'seam', 'seams', 'sewing', 'thread', 'hem'],
    'color':     ['color', 'colour', 'faded', 'vibrant', 'bright', 'dark', 'dye', 'print', 'pattern'],
    'design':    ['design', 'style', 'look', 'appearance', 'aesthetic',
                  'beautiful', 'ugly', 'elegant', 'pretty'],
    'quality':   ['quality', 'durable', 'durability', 'cheap', 'premium',
                  'well made', 'poorly made', 'broke apart'],
    'delivery':  ['delivery', 'shipping', 'arrived', 'package', 'packaging', 'shipped'],
    'price':     ['price', 'cost', 'worth', 'expensive', 'affordable', 'value', 'overpriced'],
    'zipper':    ['zipper', 'zip', 'button', 'buttons', 'closure', 'snap'],
    'comfort':   ['comfort', 'comfortable', 'uncomfortable', 'cozy', 'itchy', 'very comfortable'],
    'length':    ['length', 'long', 'short', 'maxi', 'mini', 'crop', 'ankle'],
}

def extract_aspects(text, sentiment):
    found = [a for a, kws in ASPECT_KEYWORDS.items()
             if any(kw in text.lower() for kw in kws)]
    return ', '.join(f'{a}: {sentiment}' for a in found) if found else None

df['output'] = df.apply(lambda r: extract_aspects(r['text'], r['sentiment']), axis=1)
df = df.dropna(subset=['output'])
print(f"Reviews with aspects: {len(df)}")
print(df[['text','output']].head(6).to_string())


In [ ]:
sample_per_class = 66_667   # ~200k total

df_balanced = pd.concat([
    df[df['sentiment']=='positive'].sample(min(sample_per_class,(df['sentiment']=='positive').sum()),random_state=42),
    df[df['sentiment']=='negative'].sample(min(sample_per_class,(df['sentiment']=='negative').sum()),random_state=42),
    df[df['sentiment']=='neutral' ].sample(min(sample_per_class,(df['sentiment']=='neutral' ).sum()),random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset size: {len(df_balanced)}')
print(df_balanced['sentiment'].value_counts())


## 6. Tokenise & Train / Validation / Test Split  (70 / 15 / 15)

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

MODEL_NAME = 't5-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_IN, MAX_OUT = 128, 64

def preprocess(examples):
    inputs = [f'aspect-sentiment analysis: {t}' for t in examples['text']]
    mi     = tokenizer(inputs,              max_length=MAX_IN,  padding='max_length', truncation=True)
    lbl    = tokenizer(examples['output'],  max_length=MAX_OUT, padding='max_length', truncation=True)
    mi['labels'] = [[(t if t!=tokenizer.pad_token_id else -100) for t in lab] for lab in lbl['input_ids']]
    return mi

raw_ds   = Dataset.from_pandas(df_balanced[['text','output']])
tv_split = raw_ds.train_test_split(test_size=0.30, seed=42)
vt_split = tv_split['test'].train_test_split(test_size=0.50, seed=42)

dataset = DatasetDict({
    'train':      tv_split['train'],
    'validation': vt_split['train'],
    'test':       vt_split['test'],
}).map(preprocess, batched=True, remove_columns=['text','output'])

print(f'Train      : {len(dataset["train"])}')
print(f'Validation : {len(dataset["validation"])}')
print(f'Test       : {len(dataset["test"])}')


## 7. Training

In [ ]:
from transformers import (
    T5ForConditionalGeneration, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
)
import torch

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

args = Seq2SeqTrainingArguments(
    output_dir='./t5base_clothing_v2',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    warmup_steps=200,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

print('Starting training...')
trainer.train()
print('Training complete!')


## 8. Training & Validation Loss Curves

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(e['epoch'],e['loss'])      for e in log if 'loss' in e and 'eval_loss' not in e]
eval_pts  = [(e['epoch'],e['eval_loss']) for e in log if 'eval_loss' in e]
te, tl = zip(*train_pts); ve, vl = zip(*eval_pts)

plt.figure(figsize=(9,5))
plt.plot(te, tl, 'b-o', label='Train Loss',      linewidth=2)
plt.plot(ve, vl, 'r-s', label='Validation Loss', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training vs Validation Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('loss_curves.png', dpi=150); plt.show()


## 9. Generate Predictions on Test Set

In [ ]:
from tqdm.auto import tqdm

model.eval()

# Rebuild raw test split with same seed to get original text
raw2   = Dataset.from_pandas(df_balanced[['text','output']].reset_index(drop=True))
tv2    = raw2.train_test_split(test_size=0.30, seed=42)
vt2    = tv2['test'].train_test_split(test_size=0.50, seed=42)
test_raw = vt2['test']

BATCH = 64
all_preds, all_targets, all_inputs = [], [], []

for i in tqdm(range(0, len(test_raw), BATCH), desc='Test inference'):
    b   = test_raw[i:i+BATCH]
    enc = tokenizer(
        [f'aspect-sentiment analysis: {t}' for t in b['text']],
        max_length=MAX_IN, padding=True, truncation=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_OUT, num_beams=4,
                             early_stopping=True, no_repeat_ngram_size=3,
                             repetition_penalty=2.5)
    all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    all_targets.extend(b['output'])
    all_inputs.extend(b['text'])

print(f'Generated {len(all_preds)} predictions')


## 10. ROUGE Score

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []
for p, t in zip(all_preds, all_targets):
    s = scorer.score(t, p)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)

print(f'ROUGE-1 : {sum(r1)/len(r1):.4f}')
print(f'ROUGE-2 : {sum(r2)/len(r2):.4f}')
print(f'ROUGE-L : {sum(rL)/len(rL):.4f}')


## 11. Exact Match & Token-Level Accuracy

In [ ]:
exact     = sum(p.strip()==t.strip() for p,t in zip(all_preds,all_targets))
exact_acc = exact / len(all_preds)
print(f'Exact Match Accuracy  : {exact_acc:.4f}  ({exact}/{len(all_preds)})')

def token_acc(pred, ref):
    p, r = pred.strip().split(), ref.strip().split()
    if not r: return 1.0 if not p else 0.0
    return sum(a==b for a,b in zip(p,r)) / max(len(p),len(r))

tok_accs = [token_acc(p,t) for p,t in zip(all_preds,all_targets)]
print(f'Token-Level Accuracy  : {sum(tok_accs)/len(tok_accs):.4f}')


## 12. Sentiment Accuracy & Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

SENTIMENTS = ['positive', 'negative', 'neutral']

def dom_sentiment(text):
    for s in SENTIMENTS:
        if s in text.lower(): return s
    return 'unknown'

y_true_raw = [dom_sentiment(t) for t in all_targets]
y_pred_raw = [dom_sentiment(p) for p in all_preds]
valid  = [(t,p) for t,p in zip(y_true_raw,y_pred_raw) if t!='unknown' and p!='unknown']
yt, yp = zip(*valid)

print('=== Sentiment Classification Report ===')
print(classification_report(yt, yp, labels=SENTIMENTS, zero_division=0))

sent_acc = accuracy_score(yt, yp)
sent_f1  = f1_score(yt, yp, average='weighted', labels=SENTIMENTS, zero_division=0)
print(f'Accuracy    : {sent_acc:.4f}')
print(f'Weighted F1 : {sent_f1:.4f}')

cm = confusion_matrix(yt, yp, labels=SENTIMENTS)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SENTIMENTS, yticklabels=SENTIMENTS)
plt.title('Confusion Matrix — Sentiment (Test Set)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.savefig('confusion_matrix_sentiment.png', dpi=150); plt.show()


## 13. Aspect Detection — Multi-Label Metrics & Confusion Matrix

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import hamming_loss, jaccard_score, f1_score as f1

ASPECTS = list(ASPECT_KEYWORDS.keys())

def aspects_in(text):
    tl = text.lower()
    return {a for a in ASPECTS if a in tl}

mlb    = MultiLabelBinarizer(classes=ASPECTS)
yt_asp = mlb.fit_transform([aspects_in(t) for t in all_targets])
yp_asp = mlb.transform([aspects_in(p) for p in all_preds])

hl     = hamming_loss(yt_asp, yp_asp)
jac    = jaccard_score(yt_asp, yp_asp, average='samples', zero_division=0)
f1_asp = f1(yt_asp, yp_asp, average='macro', zero_division=0)

print(f'Aspect Hamming Loss      : {hl:.4f}  (lower is better)')
print(f'Aspect Jaccard (samples) : {jac:.4f}')
print(f'Aspect Macro F1          : {f1_asp:.4f}')

# Per-aspect F1 bar chart
per_f1 = f1(yt_asp, yp_asp, average=None, zero_division=0)
asp_df = pd.DataFrame({'aspect': ASPECTS, 'f1': per_f1}).sort_values('f1', ascending=False)

plt.figure(figsize=(11,5))
plt.bar(asp_df['aspect'], asp_df['f1'], color='steelblue')
plt.title('Per-Aspect F1 Score (Test Set)')
plt.xlabel('Aspect'); plt.ylabel('F1')
plt.xticks(rotation=30, ha='right'); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('aspect_f1.png', dpi=150); plt.show()
print(asp_df.to_string(index=False))

# Aspect co-occurrence heatmap (true rows vs predicted cols)
co = yt_asp.T @ yp_asp
plt.figure(figsize=(10,8))
sns.heatmap(co, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=ASPECTS, yticklabels=ASPECTS)
plt.title('Aspect Co-Occurrence: True (rows) vs Predicted (cols)')
plt.tight_layout(); plt.savefig('confusion_matrix_aspect.png', dpi=150); plt.show()


## 14. Accuracy Curve per Epoch

In [ ]:
eval_log = [(e['epoch'],e['eval_loss']) for e in log if 'eval_loss' in e]
epochs_e = [x[0] for x in eval_log]
losses_e = [x[1] for x in eval_log]

mn, mx = min(losses_e), max(losses_e)
approx_acc = [(1-(l-mn)/(mx-mn+1e-9))*100 for l in losses_e]

fig, ax1 = plt.subplots(figsize=(9,5))
ax1.plot(epochs_e, losses_e,   'b-o', label='Val Loss',         linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Eval Loss', color='b')
ax2 = ax1.twinx()
ax2.plot(epochs_e, approx_acc, 'r-s', label='Approx Accuracy %', linewidth=2)
ax2.set_ylabel('Approx Accuracy (%)', color='r')
ax1.set_title('Validation Loss & Approximate Accuracy per Epoch')
lines  = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc='center right')
plt.tight_layout(); plt.savefig('accuracy_curve.png', dpi=150); plt.show()


## 15. Evaluation Summary

In [ ]:
print('\n' + '='*65)
print(' EVALUATION SUMMARY '.center(65))
print('='*65)
print(f'  Dataset            : 200 000 samples')
print(f'  Split              : 70 / 15 / 15  train / val / test')
print(f'  Exact Match Acc    : {exact_acc:.4f}')
print(f'  Token-Level Acc    : {sum(tok_accs)/len(tok_accs):.4f}')
print(f'  Sentiment Acc      : {sent_acc:.4f}')
print(f'  Sentiment W-F1     : {sent_f1:.4f}')
print(f'  ROUGE-1 / 2 / L   : {sum(r1)/len(r1):.4f} / {sum(r2)/len(r2):.4f} / {sum(rL)/len(rL):.4f}')
print(f'  Aspect Macro F1    : {f1_asp:.4f}')
print(f'  Aspect Jaccard     : {jac:.4f}')
print(f'  Aspect Hamming     : {hl:.4f}')
print('='*65)


## 16. Qualitative Test  *(includes slang reviews)*

In [ ]:
model.eval()

test_reviews = [
    # Standard
    'the fabric is very soft but the size runs too small',
    'zipper broke after one use, terrible quality',
    'beautiful design and perfect fit, fast delivery',
    'color faded after first wash and stitching came apart',
    'comfortable material but a bit overpriced',
    # Slang
    'tbh this is trash, fell apart after 1 wash smh',
    'omg soooo soft, tts, 10/10 would buy again',
    'mid quality, paper thin fabric, ripoff for the price',
    'slaying in this dress, fire fit and the color is lit',
    "ngl the stitching is weak af, returned it immediately",
    "goat level comfort, super comfy and design is chef's kiss",
]

print('='*70)
print('QUALITATIVE RESULTS — slang normalised before inference'.center(70))
print('='*70)
passed = 0
for i, review in enumerate(test_reviews, 1):
    cleaned = clean_text(review)
    enc = tokenizer(f'aspect-sentiment analysis: {cleaned}',
                    return_tensors='pt', max_length=MAX_IN, truncation=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_OUT, num_beams=4,
                             early_stopping=True, no_repeat_ngram_size=3,
                             repetition_penalty=2.5)
    result = tokenizer.decode(out[0], skip_special_tokens=True)
    is_valid = ':' in result and any(s in result for s in ['positive','negative','neutral'])
    if is_valid: passed += 1
    tag = '✅' if is_valid else '❌'
    print(f'\n[{i:02d}] {tag}')
    print(f'  Original : {review}')
    print(f'  Cleaned  : {cleaned}')
    print(f'  Output   : {result}')

print('\n' + '='*70)
print(f'SCORE: {passed}/{len(test_reviews)}')


## 17. Save & Download

In [ ]:
import shutil
from google.colab import files

model.save_pretrained('./t5base_clothing_v2')
tokenizer.save_pretrained('./t5base_clothing_v2')
shutil.make_archive('t5base_v2', 'zip', './t5base_clothing_v2')
files.download('t5base_v2.zip')
print('Downloaded t5base_v2.zip')
